# Lección 00_08 — Archivos e introducción a pandas

## 1. Objetivo

Leer un archivo CSV de lecturas de turno, explorar datos con pandas, filtrar por calidad y crear un gráfico de tendencia — preparación directa para el Lab 01 de PI System.

## 2. Concepto

**pandas** es la librería estándar para tablas de datos en Python. Un `DataFrame` es como una hoja de Excel en memoria: filas, columnas y operaciones vectorizadas.

### Si vienes de Excel...

`pd.read_csv('archivo.csv')` carga un CSV como DataFrame. `.head()` es como ver las primeras filas; `.describe()` como estadísticas descriptivas.

## Configuración del entorno

> **Google Colab:** Ejecuta primero la celda **Configuración del entorno**. Detecta Colab automáticamente, instala dependencias si faltan y prepara carpetas `data/` y `outputs/`.
> **Jupyter local:** La misma celda funciona con el entorno ExcelA (`uv sync`).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _in_colab()

try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"])
    import pandas as pd
    import matplotlib.pyplot as plt

if IN_COLAB:
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except NameError:
        pass

if IN_COLAB:
    _candidatos = [
        Path("/content/ExcelA/00_python_para_ingenieros"),
        Path("/content/drive/MyDrive/ExcelA/00_python_para_ingenieros"),
        Path("/content/drive/MyDrive/Colab Notebooks/ExcelA/00_python_para_ingenieros"),
        Path("/content/00_python_para_ingenieros"),
        Path.cwd(),
    ]
    MOD_DIR = next(
        (p for p in _candidatos if (p / "README_modulo0.md").exists() or (p / "data").is_dir()),
        Path("/content/00_python_para_ingenieros"),
    )
    print("Entorno: Google Colab")
else:
    MOD_DIR = Path.cwd()
    print("Entorno: Jupyter local")

MOD_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(MOD_DIR)
OUTPUT_DIR = MOD_DIR / "outputs"
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "lecturas_turno.csv"
if not DATA_PATH.exists():
    import numpy as np
    from datetime import datetime, timedelta
    rng = np.random.default_rng(42)
    start = datetime(2025, 3, 1, 8, 0)
    rows = []
    for h in range(24):
        ts = start + timedelta(hours=h)
        for var, unidad, base, noise in [("TEMP_RODAMIENTO", "°C", 70.0, 3.0), ("VIBRACION_RMS", "mm/s", 2.5, 0.3)]:
            rows.append({
                "Timestamp": ts, "Equipo": "PUMP101", "Variable": var,
                "Valor": round(base + rng.normal(0, noise), 2), "Unidad": unidad,
                "Quality": "BAD" if rng.random() < 0.03 else "GOOD",
            })
    pd.DataFrame(rows).to_csv(DATA_PATH, index=False)
    print(f"Datos generados en Colab: {DATA_PATH}")

print(f"Directorio del módulo: {MOD_DIR}")


## 3. Ejemplos guiados

In [ ]:
# Leer CSV de lecturas de turno
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df)}")
df.head()


In [ ]:
# Exploración básica
print("Columnas:", df.columns.tolist())
print("\nEstadísticas:")
df["Valor"].describe()


In [ ]:
# Filtrar solo datos con calidad GOOD (como en PI System)
df_good = df[df["Quality"] == "GOOD"]
print(f"Registros GOOD: {len(df_good)} de {len(df)}")

# Filtrar temperatura de rodamiento de PUMP101
df_temp = df_good[
    (df_good["Equipo"] == "PUMP101") & (df_good["Variable"] == "TEMP_RODAMIENTO")
]
df_temp.head()


In [ ]:
# Gráfico de tendencia de temperatura
serie = df_temp.set_index("Timestamp")["Valor"]

plt.figure(figsize=(10, 4))
plt.plot(serie.index, serie.values, marker="o", markersize=3, color="steelblue")
plt.axhline(75, color="orange", linestyle="--", label="Umbral alerta 75 °C")
plt.title("PUMP101 — Temperatura de rodamiento (turno)")
plt.xlabel("Hora")
plt.ylabel("°C")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

grafico_path = OUTPUT_DIR / "tendencia_temp_pump101.png"
plt.savefig(grafico_path, dpi=150)
print(f"Gráfico guardado: {grafico_path}")


In [ ]:
# Exportar resumen a CSV
resumen = df_good.groupby(["Equipo", "Variable"])["Valor"].agg(["mean", "max", "min"]).reset_index()
resumen_path = OUTPUT_DIR / "resumen_turno.csv"
resumen.to_csv(resumen_path, index=False)
print(f"Resumen exportado: {resumen_path}")
resumen


## 5. Ejercicio práctico — Filtrar vibración crítica

Filtra registros GOOD de PUMP101 con variable VIBRACION_RMS y calcula el máximo. Si supera 4.5 mm/s, imprime una alerta.

In [ ]:
df_vib = df_good[(df_good["Equipo"]=="PUMP101") & (df_good["Variable"]=="VIBRACION_RMS")]
max_vib = df_vib["Valor"].max()
print(f"Vibración máxima: {max_vib} mm/s")
if max_vib > 4.5:
    print("ALERTA: vibración supera umbral de alerta")

## 6. Resumen y siguiente paso

- `pd.read_csv()` carga datos tabulares.
- Filtrar con `df[condicion]` es equivalente a filtros en Excel.
- Ya puedes leer exports PI y graficar tendencias.

**Siguiente lección:** **Lab 01 — `01_PI_historiador_introduccion.ipynb`** (siguiente módulo del curso)